# Context Managers & Resource Management: Beginner Guide

### 🌟 What Are Context Managers & Resource Management?
**Context Managers** (used with the `with` keyword) automate the setup and cleanup of system resources like database connections, open files, and thread locks. Using the `__enter__` and `__exit__` protocol prevents resource leaks in long-running services.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Class Context Protocol**: Covers `__enter__()` and `__exit__()` (handling exceptions and suppression).
- **Generator Context**: Covers `@contextlib.contextmanager` and `try...finally`.
- **Context Utilities**: Covers `contextlib.suppress()`, `contextlib.redirect_stdout()`, and `contextlib.ExitStack`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get('transaction_amount'):
            transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 14635 transaction records from ../data/raw_transactions.csv


### 🔹 Class Context Protocol: `__enter__()`
Acquires resource and returns object target bound to `as target` variable. Object-Oriented Programming groups related data and functions together, making code modular, maintainable, and easy to scale. **Tip:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.

**Syntax:** `def __enter__(self): return resource`


In [2]:
class DBTransaction:
    def __enter__(self):
        print('[DB] BEGIN TRANSACTION')
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print('[DB] COMMIT TRANSACTION')
        return False

with DBTransaction():
    print('Executing database queries...')

[DB] BEGIN TRANSACTION
Executing database queries...
[DB] COMMIT TRANSACTION


### 🔹 Class Context Protocol: `__exit__()` & Exception Suppression
Releases resource. Returning `True` swallows exceptions; returning `False`/`None` propagates exceptions. Object-Oriented Programming groups related data and functions together, making code modular, maintainable, and easy to scale. **Tip:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.

**Syntax:** `def __exit__(self, exc_type, exc_val, exc_tb): return True`


In [3]:
class SafeExecutionLock:
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f'[SUPPRESSED ERROR] {exc_val}')
            return True # Suppress exception
        return False

with SafeExecutionLock():
    raise KeyError('Simulated missing key (safely suppressed)')

[SUPPRESSED ERROR] 'Simulated missing key (safely suppressed)'


### 🔹 Generator Context Manager: `@contextlib.contextmanager`
Transforms generator function with a single `yield` into a context manager. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `@contextlib.contextmanager def manager(): ... yield ...`


In [4]:
@contextlib.contextmanager
def execution_timer(label):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        print(f'{label} elapsed: {(time.perf_counter()-t0)*1000:.3f} ms')

with execution_timer('Scan 5k records'):
    s = sum(float(t['transaction_amount'] or 0.0) for t in transactions[:5000])
    print(f'Sum: ${s:,.2f}')

Sum: $nan
Scan 5k records elapsed: 0.878 ms


### 🔹 Context Utility: `contextlib.suppress()`
Cleanly ignores specified exception classes within context scope. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `with contextlib.suppress(FileNotFoundError): ...`


In [5]:
with contextlib.suppress(FileNotFoundError):
    os.remove('non_existent_file.tmp')
print('Suppressed FileNotFoundError cleanly.')

Suppressed FileNotFoundError cleanly.


### 🔹 Context Utility: `contextlib.redirect_stdout()`
Redirects standard stdout stream to an in-memory buffer. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `with contextlib.redirect_stdout(buf): ...`


In [6]:
import io
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print('Hidden transaction audit message')
print('Captured redirected output:', repr(buf.getvalue().strip()))

Captured redirected output: 'Hidden transaction audit message'


### 🔹 Dynamic Context Stacking: `contextlib.ExitStack`
Programmatically opens and manages dynamic variable numbers of context managers. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `with ExitStack() as stack: ...`


In [7]:
with contextlib.ExitStack() as stack:
    f1 = stack.enter_context(open(csv_path, 'r'))
    print('ExitStack opened file safely. First line length:', len(f1.readline()))

ExitStack opened file safely. First line length: 151


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Re-entrant vs Single-Use Context Managers

**Approach:** Explain how context managers must reset their internal state in `__enter__` to support multiple re-entries.
**Syntax:** `class ReentrantContext: ...`


In [8]:
print('Re-entrant context managers reset state in __enter__ to allow nested or looped with blocks.')

Re-entrant context managers reset state in __enter__ to allow nested or looped with blocks.
